# Inspection, Debugging & File I/O

This notebook covers tools for understanding, validating, and persisting your networks:

- `inspect()` — automated network validation
- DataFrame inspection and query helpers
- UUID ↔ tuple conversion
- File I/O: JSON and Excel
- Logging

## Setup

In [1]:
import pandaict as pi
from pandaict.utils.enums import NodeType, DeviceType, AppType, InfoItem

net = pi.create_empty_network(name='Inspection Demo')

n_core  = pi.create_ict_node(net, name='Core-Router',   node_type=NodeType.ROUTER, link_types=['Fiber','LAN1000'])
n_sw1   = pi.create_ict_node(net, name='Switch-1',      node_type=NodeType.SWITCH, link_types=['LAN1000'])
n_sw2   = pi.create_ict_node(net, name='Switch-2',      node_type=NodeType.SWITCH, link_types=['LAN1000'])

ctrl_server = pi.create_ict_device(net, name='CTRL-Server', device_type=DeviceType.SERVER)
rtu_a       = pi.create_ict_device(net, name='RTU-A',       device_type=DeviceType.RTU)

pi.create_ict_link(net, name='L-SW1-Core',   from_elt=n_sw1,        to_elt=[n_core],       link_type='LAN1000')
pi.create_ict_link(net, name='L-Core-SW2',   from_elt=n_core,       to_elt=[n_sw2],        link_type='LAN1000')
pi.create_ict_link(net, name='L-Core-CTRL',  from_elt=n_core,       to_elt=[ctrl_server],  link_type='LAN1000')
pi.create_ict_link(net, name='L-SW2-RTUA',   from_elt=n_sw2,        to_elt=[rtu_a],        link_type='LAN1000')
pi.create_ict_link(net, name='L-SW1-SW2',    from_elt=n_sw1,        to_elt=[n_sw2],        link_type='LAN1000')

app_ctrl = pi.create_ict_app(net, name='App-CTRL',  app_type=AppType.CLIENT,    device=ctrl_server)
app_rtua = pi.create_ict_app(net, name='App-RTU-A', app_type=AppType.WEBSERVER, device=rtu_a)

pi.create_ict_connection(net, name='CTRL-to-RTU-A',
    from_app=app_ctrl, to_app=app_rtua, protocol='TCP',
    info_items=[InfoItem.REQUEST, InfoItem.VALUE],
    peers=(app_ctrl, app_rtua), producer=[0, 1])

print('Demo network ready.')

Demo network ready.


## Automated inspection with `inspect()`

The `inspect()` function walks through all elements and checks for common mistakes: missing names, missing types, nodes with fewer than two links, unassigned apps, etc.

In [2]:
passed = pi.inspect(net, verbose=True)
print('\nAll checks passed:', passed)

----------- INSPECTION PASSED -----------

All checks passed: True


With `verbose=False`, the function runs silently and only returns the boolean result:

In [3]:
ok = pi.inspect(net, verbose=False)
print('OK:', ok)

----------- INSPECTION PASSED -----------
OK: True


### Deliberately break something to see what happens

In [4]:
# Force a type to be None (don't do this at home!)
net.node.at[0, 'type'] = None
pi.inspect(net, verbose=True)   # Will log the issue

node 'Core-Router' (index 0) has no type
----------- INSPECTION FAILED -----------


False

In [5]:
# Restore it
net.node.at[0, 'type'] = NodeType.ROUTER

## DataFrame inspection

Since every element type is stored as a pandas DataFrame, you can use all standard pandas operations:

In [6]:
print('--- Nodes ---')
display(net.node[['name', 'type', 'link_types']])

print('\n--- Links ---')
display(net.link[['name', 'type', 'from', 'to']])

print('\n--- Apps ---')
display(net.app[['name', 'type', 'device']])

print('\n--- Connections ---')
display(net.conn[['name', 'protocol', 'from', 'to']])

--- Nodes ---


,name,type,link_types
0,Core-Router,NodeType.ROUTER,"[Fiber, LAN1000]"
1,Switch-1,NodeType.SWITCH,[LAN1000]
2,Switch-2,NodeType.SWITCH,[LAN1000]



--- Links ---


,name,type,from,to
0,L-SW1-Core,LAN1000,"(node, 1)","[(node, 0)]"
1,L-Core-SW2,LAN1000,"(node, 0)","[(node, 2)]"
2,L-Core-CTRL,LAN1000,"(node, 0)","[(device, 0)]"
3,L-SW2-RTUA,LAN1000,"(node, 2)","[(device, 1)]"
4,L-SW1-SW2,LAN1000,"(node, 1)","[(node, 2)]"



--- Apps ---


,name,type,device
0,App-CTRL,AppType.CLIENT,0
1,App-RTU-A,AppType.WEBSERVER,1



--- Connections ---


,name,protocol,from,to
0,CTRL-to-RTU-A,TCP,0,1


## Element query helpers

pandaict provides utility functions to query the network graph:

In [7]:
from pandaict.utils.utils import get_links_at, get_apps_on_device, get_links_between, get_elt_id_by_name

# Links connected to Core-Router
links_at_core = get_links_at(net, n_core)
for l in links_at_core:
    print('Link at Core-Router:', net.link.loc[l]['name'])

print()
# Apps on CTRL-Server
apps = get_apps_on_device(net, ctrl_server)
for a in apps:
    print('App on CTRL-Server:', net.app.loc[a]['name'])

print()
# Find element by name
print('get_elt_id_by_name:', get_elt_id_by_name(net, 'Core-Router'))

Link at Core-Router: L-SW1-Core
Link at Core-Router: L-Core-SW2
Link at Core-Router: L-Core-CTRL

App on CTRL-Server: App-CTRL

get_elt_id_by_name: ('node', np.int64(0))


## UUID <-> tuple conversion

If you use UUIDs for element identification, these helpers bridge between the two systems:

In [8]:
from pandaict.utils.utils import uuid2elt, elt2uuid

# Get the UUID of Core-Router (('node', 0))
uid = elt2uuid(net, n_core)
print('UUID of Core-Router:', uid)

# Convert back to tuple
back = uuid2elt(net, uid)
print('Back to tuple:      ', back)

UUID of Core-Router: None
Back to tuple:       None


## File I/O: JSON

pandaict supports JSON serialization with full fidelity (preserves UUIDs, custom enums, std_types):

In [9]:
# Write to JSON
pi.to_json(net, 'demo_network.json')
print('Network saved to demo_network.json')

Network saved to demo_network.json


In [10]:
# Read back
loaded = pi.from_json('demo_network.json')
print('Loaded network:', loaded.name)
print('Nodes:', len(loaded.node), '| Links:', len(loaded.link))
display(loaded.node[['name', 'type']])

Loaded network: Inspection Demo
Nodes: 3 | Links: 5


,name,type
0,Core-Router,NodeType.ROUTER
1,Switch-1,NodeType.SWITCH
2,Switch-2,NodeType.SWITCH


In [11]:
# Clean up
import os
os.remove('demo_network.json')

## File I/O: Excel

The `toolbox.exceltools` module provides Excel import/export with a structured workbook format:

In [12]:
from pandaict.toolbox.exceltools import to_excel, from_excel

# Export
to_excel(net, 'demo_network.xlsx')
print('Exported to demo_network.xlsx')

# Import
loaded = from_excel('demo_network.xlsx')
print('Import done.')

os.remove('demo_network.xlsx')

Exported to demo_network.xlsx


UserWarning: Can not convert 'node, np.int641' into <class 'tuple'> for from.

## Logging

pandaict logs to `pandaICT_Core.log` via the `pilogger` module. All warnings and debug output from routing, validation, and reliability go there. In Jupyter, you might not see a file handler; standalone scripts will produce the log file.

You can also access the logger directly:

In [ ]:
from pandaict.pilogger import pilogger

pilogger.info('Demonstration log message from notebook 05.')
print('Log level:', pilogger.level)
print('Log handlers:', pilogger.handlers)

## Summary

| Tool | Purpose |
|------|---------|
| `inspect()` | Automated network validation |
| DataFrames (`net.node`, `net.link`, ...) | Direct element inspection |
| `get_links_at()`, `get_apps_on_device()`, etc. | Graph queries |
| `uuid2elt()` / `elt2uuid()` | UUID ↔ tuple conversion |
| `to_json()` / `from_json()` | JSON serialization |
| `to_excel()` / `from_excel()` | Excel import/export |
| `pilogger` | Diagnostic logging |

Next: [06 – Standard Types & Custom Enums](06_std_types.ipynb)